<a href="https://colab.research.google.com/github/Advenarrow/Projects-on-digital-logic-design-using-AI/blob/main/UART_Baud_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Icarus Verilog Installation and Verification

Before we can simulate our Verilog designs, we need to ensure that Icarus Verilog (a Verilog compiler and simulator) is installed on the Colab environment. The following steps will:

1.  **Install Icarus Verilog:** The `!apt-get install -y iverilog` command uses the `apt-get` package manager to install the necessary tools.
2.  **Verify Installation:** The `!iverilog -v` command checks the installed version of Icarus Verilog, confirming that it's correctly set up and ready for use.

In [6]:
!apt-get install -y iverilog
!iverilog -v

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
Icarus Verilog version 11.0 (stable) ()

Copyright 1998-2020 Stephen Williams

  This program is free software; you can redistribute it and/or modify
  it under the terms of the GNU General Public License as published by
  the Free Software Foundation; either version 2 of the License, or
  (at your option) any later version.

  This program is distributed in the hope that it will be useful,
  but WITHOUT ANY WARRANTY; without even the implied warranty of
  MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
  GNU General Public License for more details.

  You should have received a copy of the GNU General Public License along
  with this program; if not, write to the Free Software Foundation, Inc.,
  51 Franklin Street, Fifth Floor, Boston, MA 02110-1301, USA.

iveril

### `baud_gen` Module Design

This section defines the Verilog module for the `baud_gen` (Baud Rate Generator). The purpose of this module is to generate a `tick` signal at a frequency determined by the `DIV` parameter, effectively creating a baud clock.

**Module Parameters and Ports:**
*   `DIV`: A parameter that determines the division factor. The `tick` signal will assert once every `DIV` clock cycles.
*   `clk`: The input clock signal.
*   `rst`: The asynchronous reset signal (active high).
*   `tick`: The output signal, which goes high for one clock cycle when the division count is reached.

**Internal Logic:**
*   `count`: A 10-bit register to keep track of the clock cycles. It counts from 0 up to `DIV - 1`.
*   **`always @(posedge clk)` block:** This describes the sequential logic of the module, operating on the rising edge of the clock.
    *   **Reset Condition:** When `rst` is high, `count` is reset to 0, and `tick` is de-asserted (0).
    *   **Division Condition:** If `count` reaches `DIV - 1`, it means `DIV` clock cycles have passed. In this case, `count` is reset to 0, and `tick` is asserted (1) for one clock cycle.
    *   **Counting:** Otherwise, `count` increments by 1, and `tick` remains de-asserted (0).

In [7]:
%%writefile baud_gen.v

module baud_gen #(
    parameter DIV = 5
)(
    input       clk,
    input       rst,
    output reg  tick
);

reg [9:0] count;

always @(posedge clk) begin
    if (rst) begin
        count <= 0;
        tick  <= 0;
    end else if (count == DIV - 1) begin
        count <= 0;
        tick  <= 1;
    end else begin
        count <= count + 1;
        tick  <= 0;
    end
end

endmodule

Overwriting baud_gen.v


### Testbench for `baud_gen`

To verify the functionality of our `baud_gen` module, we need a testbench. A testbench is a Verilog module that instantiates the design under test (DUT) – in this case, `baud_gen` – and provides it with input stimuli (like clock and reset signals) while monitoring its outputs.

This `baud_gen_tb.v` testbench includes the following key components:

*   **Signal Declarations:** `reg` for inputs (controlled by the testbench) and `wire` for outputs (driven by the DUT).
*   **DUT Instantiation:** The `baud_gen` module is instantiated as `uut` (unit under test). The `DIV` parameter is set to `5` initially to generate a tick every 5 clock cycles.
*   **Clock Generation:** An `always #5 clk = ~clk;` block creates a continuous clock signal with a period of 10 time units (5 time units high, 5 time units low).
*   **Initial Block:** This block defines the simulation sequence:
    *   `$dumpfile("dump.vcd");` and `$dumpvars(0, baud_gen_tb);` are used to enable Value Change Dump (VCD) file generation, which allows us to view waveforms in tools like GTKWave.
    *   Initializes `clk` to 0 and `rst` to 1 (active high reset).
    *   Holds `rst` high for a few clock cycles to ensure proper reset.
    *   De-asserts `rst` and lets the simulation run for 50 clock cycles.
    *   `$display("Simulation complete");` and `$finish;` mark the end of the simulation.
*   **Tick Detection and Display:** An `always @(posedge clk)` block checks for the `tick` signal and displays the simulation time when a tick occurs.
*   **Monitor:** An `initial $monitor` statement continuously displays the values of `clk`, `rst`, and `tick` along with the current simulation time, providing a textual log of the simulation.

In [8]:
%%writefile baud_gen_tb.v

module baud_gen_tb;

reg  clk, rst;
wire tick;

baud_gen #(.DIV(5)) uut (
    .clk  (clk),
    .rst  (rst),
    .tick (tick)
);

always #5 clk = ~clk;

initial begin
    $dumpfile("dump.vcd");
    $dumpvars(0, baud_gen_tb);
    clk = 0; rst = 1;
    repeat(3) @(posedge clk);
    rst = 0;
    repeat(50) @(posedge clk);
    $display("Simulation complete");
    $finish;
end

always @(posedge clk) begin
    if (tick)
        $display("TICK at time = %0t ns", $time);
end

initial
    $monitor("t=%0t clk=%b rst=%b tick=%b",
              $time, clk, rst, tick);

endmodule

Overwriting baud_gen_tb.v


### Compile and Simulate the Verilog Design

Now that we have defined both the `baud_gen` module and its `baud_gen_tb` testbench, we can compile and simulate them using Icarus Verilog.

1.  **Compilation (`iverilog`):**
    *   `!iverilog -o baud_sim baud_gen.v baud_gen_tb.v`:
        *   The `iverilog` command is the Icarus Verilog compiler.
        *   `-o baud_sim` specifies that the output executable simulation file should be named `baud_sim`.
        *   `baud_gen.v` and `baud_gen_tb.v` are the source Verilog files to be compiled.
    *   This step translates our human-readable Verilog code into an executable format that the simulator can understand.

2.  **Simulation (`vvp`):**
    *   `!vvp baud_sim`:
        *   The `vvp` command is the Icarus Verilog simulator.
        *   It executes the compiled simulation file (`baud_sim`).
        *   During simulation, the testbench will apply stimuli to the `baud_gen` module, and the `$monitor` and `$display` statements within the testbench will output the signal values to the console.
        *   Additionally, the simulation will generate a `dump.vcd` file, which records all signal changes over time for waveform viewing.

In [9]:
!iverilog -o baud_sim baud_gen.v baud_gen_tb.v
!vvp baud_sim

VCD info: dumpfile dump.vcd opened for output.
t=0 clk=0 rst=1 tick=x
t=5 clk=1 rst=1 tick=0
t=10 clk=0 rst=1 tick=0
t=15 clk=1 rst=1 tick=0
t=20 clk=0 rst=1 tick=0
t=25 clk=1 rst=0 tick=0
t=30 clk=0 rst=0 tick=0
t=35 clk=1 rst=0 tick=0
t=40 clk=0 rst=0 tick=0
t=45 clk=1 rst=0 tick=0
t=50 clk=0 rst=0 tick=0
t=55 clk=1 rst=0 tick=0
t=60 clk=0 rst=0 tick=0
t=65 clk=1 rst=0 tick=1
t=70 clk=0 rst=0 tick=1
TICK at time = 75 ns
t=75 clk=1 rst=0 tick=0
t=80 clk=0 rst=0 tick=0
t=85 clk=1 rst=0 tick=0
t=90 clk=0 rst=0 tick=0
t=95 clk=1 rst=0 tick=0
t=100 clk=0 rst=0 tick=0
t=105 clk=1 rst=0 tick=0
t=110 clk=0 rst=0 tick=0
t=115 clk=1 rst=0 tick=1
t=120 clk=0 rst=0 tick=1
TICK at time = 125 ns
t=125 clk=1 rst=0 tick=0
t=130 clk=0 rst=0 tick=0
t=135 clk=1 rst=0 tick=0
t=140 clk=0 rst=0 tick=0
t=145 clk=1 rst=0 tick=0
t=150 clk=0 rst=0 tick=0
t=155 clk=1 rst=0 tick=0
t=160 clk=0 rst=0 tick=0
t=165 clk=1 rst=0 tick=1
t=170 clk=0 rst=0 tick=1
TICK at time = 175 ns
t=175 clk=1 rst=0 tick=0
t=180 clk=

### Download Simulation Waveforms (VCD File)

After successfully compiling and simulating the Verilog code, a `dump.vcd` file is generated. This file contains the Value Change Dump (VCD) information, which can be used to visualize the waveform of signals during the simulation.

To view these waveforms, you typically use a waveform viewer like GTKWave (available on most operating systems). The following cell will download the generated `dump.vcd` file from the Colab environment to your local machine, allowing you to open it in your preferred waveform viewer.

In [11]:
from google.colab import files
files.download('dump.vcd')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>